In [18]:
import pandas as pd
import glob

In [6]:
for gene in pd.read_csv("../data/SFARI_TF_gene_names.tsv", sep = "\t")["Gene Names"].str.split(" ").str[0].drop_duplicates():
    print(gene)

MEIS2
KLF7
CAMTA2
NKX2-2
AR
THRA
ERG
VDR
TCF4
TFE3
NR1D1
YY1
PAX6
OTX1
RORA
MSX2
NR4A2
PITX1
PAX5
KMT2A
MEF2C
EGR3
NFIA
IKZF1
SMAD4
MTF1
NFIX
NCOA1
HIVEP3
TET2
CC2D1A
SRCAP
CASZ1
GLIS1
ESR2
ARX
KLF16
EBF3
ARNT2
TCF7L2
KDM5B
TCF20
MYT1L
TBX22
NFE2L3


Used this query: https://gpf.sfari.org/hg38/load-query/1335a41c-e716-44dd-adab-89f192febf46

To get de novo missense substitution variants in SFARI TFs (printed above) which are present only in probands and not their siblings.

In [7]:
de_novo_vars_all_cols = pd.read_csv("../data/SFARI_TF_de_novo_variants_gpf.tsv", sep="\t")
de_novo_vars = de_novo_vars_all_cols[["CHROM", "POS", "REF", "ALT"]] 
de_novo_vars["CHROM"] = de_novo_vars["CHROM"].str.split("chr").str[1]
de_novo_vars["start"] = de_novo_vars["POS"] - 1
de_novo_vars["af"] = 0
de_novo_vars = de_novo_vars[["CHROM", "start", "POS", "REF", "ALT", "af"]]
de_novo_vars

/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_26001/2890784154.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  de_novo_vars["CHROM"] = de_novo_vars["CHROM"].str.split("chr").str[1]
/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_26001/2890784154.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  de_novo_vars["start"] = de_novo_vars["POS"] - 1
/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_26001/2890784154.py:5: SettingWithCopyWarning: 
A value is trying to be set on 

,CHROM,start,POS,REF,ALT,af
0,1,61352563,61352564,C,T,0
1,2,1917306,1917307,A,G,0
2,2,156329363,156329364,A,G,0
3,1,61352563,61352564,C,T,0
4,4,105234730,105234731,G,T,0
...,...,...,...,...,...,...
112,18,55228884,55228885,G,A,0
113,18,55228993,55228994,G,A,0
114,21,38445460,38445461,G,C,0
115,22,42213188,42213189,C,T,0


In [8]:
# Make bed file for enrichment analysis scripts

In [9]:
de_novo_vars.to_csv("../soto_analysis/raw_files/SFARI_TF_de_novo_variants_gpf.bed", header = None, index = 0, sep = "\t")
de_novo_vars

,CHROM,start,POS,REF,ALT,af
0,1,61352563,61352564,C,T,0
1,2,1917306,1917307,A,G,0
2,2,156329363,156329364,A,G,0
3,1,61352563,61352564,C,T,0
4,4,105234730,105234731,G,T,0
...,...,...,...,...,...,...
112,18,55228884,55228885,G,A,0
113,18,55228993,55228994,G,A,0
114,21,38445460,38445461,G,C,0
115,22,42213188,42213189,C,T,0


In [12]:
gnomad_vars = pd.read_csv("../soto_analysis/raw_files/gnomad_missense_snps.bed", sep="\t", header=None)
gnomad_vars.columns = ['CHROM', 'start', 'POS', 'REF', 'ALT', 'af']
gnomad_vars

,CHROM,start,POS,REF,ALT,af
0,2,207081140,207081141,T,C,3.216086e-06
1,2,207081144,207081145,G,C,7.777610e-07
2,2,207081145,207081146,C,G,7.714145e-07
3,2,207081146,207081147,C,T,1.529487e-06
4,2,207081147,207081148,C,T,5.777024e-06
...,...,...,...,...,...,...
107567,19,13094705,13094706,G,A,7.334088e-07
107568,19,13094708,13094709,G,T,1.997268e-06
107569,19,13094710,13094711,A,G,1.250769e-06
107570,19,13094717,13094718,A,G,3.763805e-05


In [16]:
# Is there any overlap between the de novo vars and gnomad?

de_novo_gnomad_overlap = pd.merge(de_novo_vars, gnomad_vars, on = ['CHROM', 'start', 'POS', 'REF', 'ALT'])
display(de_novo_gnomad_overlap)
print(len(de_novo_gnomad_overlap))

,CHROM,start,POS,REF,ALT,af_x,af_y
0,1,61352563,61352564,C,T,0,3.732323e-06
1,1,61352563,61352564,C,T,0,3.732323e-06
2,9,37033985,37033986,C,G,0,6.850882e-07
3,10,129840881,129840882,C,G,0,1.591090e-06
4,16,30724602,30724603,A,T,0,6.840441e-07
5,4,105275517,105275518,C,T,0,1.429206e-06
6,10,129840881,129840882,C,G,0,1.591090e-06
7,16,30724602,30724603,A,T,0,6.840441e-07
8,16,30738521,30738522,C,T,0,7.264674e-04
9,1,202749125,202749126,C,T,0,4.983461e-06


55


In [23]:
# De novo in ADs
de_novo_variants_domains = glob.glob("../soto_analysis/outputs/mutations/domains_SFARI_TF_de_novo_variants_gpf_snv_classified/*")
domain_dfs = [pd.read_csv(file, sep="\t", header=None) for file in de_novo_variants_domains if sum(1 for line in open(file)) > 0]
domain_df = pd.concat(domain_dfs)
AD_de_novos = domain_df[domain_df[3] == "AD"]
AD_de_novos

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,17,40097201,40097202,AD,NaN,.,-1,ENST00000246672,17,40097201,...,-1,17,40097201,40097202,T,C,0,D,G,No-Syn
0,8,22692853,22692854,AD,NaN,.,-1,ENST00000317216,8,22692853,...,-1,8,22692853,22692854,T,C,0,S,G,No-Syn
3,22,42214836,42214837,AD,NaN,.,-1,ENST00000359486,22,42214836,...,-1,22,42214836,42214837,T,C,0,T,A,No-Syn
0,X,49030201,49030202,AD,NaN,.,-1,ENST00000315869,X,49030201,...,-1,X,49030201,49030202,C,A,0,A,S,No-Syn
0,5,135028999,135029000,AD,NaN,.,-1,ENST00000265340,5,135028999,...,-1,5,135028999,135029000,G,A,0,L,F,No-Syn
1,5,135028999,135029000,AD,NaN,.,-1,ENST00000265340,5,135028999,...,-1,5,135028999,135029000,G,A,0,L,F,No-Syn
1,15,36892299,36892300,AD,NaN,.,-1,ENST00000561208,15,36892299,...,-1,15,36892299,36892300,G,C,0,P,R,No-Syn
1,14,100277466,100277467,AD,NaN,.,1,ENST00000262238,14,100277466,...,1,14,100277466,100277467,G,A,0,R,H,No-Syn
2,14,100277489,100277490,AD,NaN,.,1,ENST00000262238,14,100277489,...,1,14,100277489,100277490,G,A,0,G,R,No-Syn
4,2,1922902,1922903,AD,NaN,.,-1,ENST00000428368,2,1922902,...,-1,2,1922902,1922903,A,G,0,M,T,No-Syn


In [28]:
AD_de_novos = AD_de_novos.rename(columns = {0: 'CHROM', 1: 'start', 2: 'POS', 16: 'REF', 17: 'ALT'})
AD_de_novos[['CHROM', 'start', 'POS', 'REF', 'ALT']]

,CHROM,start,POS,REF,ALT
0,17,40097201,40097202,T,C
0,8,22692853,22692854,T,C
3,22,42214836,42214837,T,C
0,X,49030201,49030202,C,A
0,5,135028999,135029000,G,A
1,5,135028999,135029000,G,A
1,15,36892299,36892300,G,C
1,14,100277466,100277467,G,A
2,14,100277489,100277490,G,A
4,2,1922902,1922903,A,G


In [36]:
gnomad_vars[['CHROM', 'start', 'POS', 'REF', 'ALT']]

,CHROM,start,POS,REF,ALT
0,2,207081140,207081141,T,C
1,2,207081144,207081145,G,C
2,2,207081145,207081146,C,G
3,2,207081146,207081147,C,T
4,2,207081147,207081148,C,T
...,...,...,...,...,...
107567,19,13094705,13094706,G,A
107568,19,13094708,13094709,G,T
107569,19,13094710,13094711,A,G
107570,19,13094717,13094718,A,G


In [ ]:
# None of 30 AD variants in SFARI TFs overlaps a gnomad variant
pd.merge(AD_de_novos, gnomad_vars, on = ['CHROM', 'start', 'POS', 'REF', 'ALT'])

,CHROM,start,POS,3,4,5,6,7,8,9,...,13,14,15,REF,ALT,18,19,20,21,af


In [ ]:
# One of 34 DBD variants in SFARI TFs overlaps a gnomad variant
DBD_de_novos = domain_df[domain_df[3] == "DBD"]
DBD_de_novos = DBD_de_novos.rename(columns = {0: 'CHROM', 1: 'start', 2: 'POS', 16: 'REF', 17: 'ALT'})
pd.merge(DBD_de_novos, gnomad_vars, on = ['CHROM', 'start', 'POS', 'REF', 'ALT'])

,CHROM,start,POS,3,4,5,6,7,8,9,...,13,14,15,REF,ALT,18,19,20,21,af
0,X,80024157,80024158,DBD,NaN,.,1,ENST00000373294,X,80024157,...,X,80024157,80024158,G,A,0,R,H,No-Syn,0.000003


# Attempt 1 below

In [19]:
de_novo_vars["effect details"].iloc[0]

'NM_001134673_1:NFIA:missense:272/509(Thr->Met)|NM_005595_1:NFIA:missense:272/498(Thr->Met)|NM_001145511_1:NFIA:missense:264/501(Thr->Met)|NM_001145512_1:NFIA:missense:317/554(Thr->Met)'

In [20]:
de_novo_vars["effect details"] = de_novo_vars["effect details"].str.split("|")
de_novo_vars["effect details"].iloc[0]

/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_4470/610310764.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  de_novo_vars["effect details"] = de_novo_vars["effect details"].str.split("|")


['NM_001134673_1:NFIA:missense:272/509(Thr->Met)',
 'NM_005595_1:NFIA:missense:272/498(Thr->Met)',
 'NM_001145511_1:NFIA:missense:264/501(Thr->Met)',
 'NM_001145512_1:NFIA:missense:317/554(Thr->Met)']

In [ ]:
de_novo_vars_expanded = de_novo_vars.explode("effect details")
de_novo_vars_expanded["full len"] = de_novo_vars_expanded["effect details"].str.extract(r"\/(.*)\(")
de_novo_vars_expanded[de_novo_vars_expanded.isna().any(axis=1)]

,CHROM,POS,REF,ALT,effect details,genes,full len
4,chr4,105234731,G,T,NR_126420_1:TET2-AS1:non-coding-intron:None/No...,TET2,NaN
5,chr9,37033986,C,G,NM_001280551_1:PAX5:5'UTR:113,PAX5,NaN
5,chr9,37033986,C,G,NM_001280556_1:PAX5:5'UTR:113,PAX5,NaN
5,chr9,37033986,C,G,NR_103999_1:PAX5:non-coding:None,PAX5,NaN
5,chr9,37033986,C,G,NR_104000_1:PAX5:non-coding:None,PAX5,NaN
8,chr4,105275518,C,T,NR_126420_1:TET2-AS1:non-coding-intron:None/No...,TET2,NaN
14,chr4,105236334,G,A,NR_126420_1:TET2-AS1:non-coding-intron:None/No...,TET2,NaN
15,chr4,105243760,G,C,NR_126420_1:TET2-AS1:non-coding-intron:None/No...,TET2,NaN
17,chr15,36950345,G,A,NR_051953_1:MEIS2:non-coding-intron:None/None[...,MEIS2,NaN
25,chr4,105275518,C,T,NR_126420_1:TET2-AS1:non-coding-intron:None/No...,TET2,NaN


In [30]:
de_novo_coding_vars_expanded = de_novo_vars_expanded.dropna()
de_novo_coding_vars_expanded

,CHROM,POS,REF,ALT,effect details,genes,full len
0,chr1,61352564,C,T,NM_001134673_1:NFIA:missense:272/509(Thr->Met),NFIA,509
0,chr1,61352564,C,T,NM_005595_1:NFIA:missense:272/498(Thr->Met),NFIA,498
0,chr1,61352564,C,T,NM_001145511_1:NFIA:missense:264/501(Thr->Met),NFIA,501
0,chr1,61352564,C,T,NM_001145512_1:NFIA:missense:317/554(Thr->Met),NFIA,554
1,chr2,1917307,A,G,NM_001303052_1:MYT1L:missense:506/1186(Cys->Arg),MYT1L,1186
...,...,...,...,...,...,...,...
114,chr21,38445461,G,C,NM_001291391_1:ERG:missense:67/325(Pro->Arg),ERG,325
115,chr22,42213189,C,T,NM_005650_3:TCF20:missense:706/1960(Arg->His),TCF20,1960
115,chr22,42213189,C,T,NM_181492_1:TCF20:missense:706/1938(Arg->His),TCF20,1938
116,chr22,42214837,T,C,NM_005650_3:TCF20:missense:157/1960(Thr->Ala),TCF20,1960


In [31]:
de_novo_coding_vars_expanded["full len"] = de_novo_coding_vars_expanded["full len"].str.strip()
de_novo_coding_vars_expanded["full len"] = de_novo_coding_vars_expanded["full len"].astype(int)
de_novo_coding_vars_expanded

/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_4470/2648747921.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  de_novo_coding_vars_expanded["full len"] = de_novo_coding_vars_expanded["full len"].str.strip()
/var/folders/hw/xx051vr9457c7lrngf2mypgr0000gn/T/ipykernel_4470/2648747921.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  de_novo_coding_vars_expanded["full len"] = de_novo_coding_vars_expanded["full len"].astype(int)


,CHROM,POS,REF,ALT,effect details,genes,full len
0,chr1,61352564,C,T,NM_001134673_1:NFIA:missense:272/509(Thr->Met),NFIA,509
0,chr1,61352564,C,T,NM_005595_1:NFIA:missense:272/498(Thr->Met),NFIA,498
0,chr1,61352564,C,T,NM_001145511_1:NFIA:missense:264/501(Thr->Met),NFIA,501
0,chr1,61352564,C,T,NM_001145512_1:NFIA:missense:317/554(Thr->Met),NFIA,554
1,chr2,1917307,A,G,NM_001303052_1:MYT1L:missense:506/1186(Cys->Arg),MYT1L,1186
...,...,...,...,...,...,...,...
114,chr21,38445461,G,C,NM_001291391_1:ERG:missense:67/325(Pro->Arg),ERG,325
115,chr22,42213189,C,T,NM_005650_3:TCF20:missense:706/1960(Arg->His),TCF20,1960
115,chr22,42213189,C,T,NM_181492_1:TCF20:missense:706/1938(Arg->His),TCF20,1938
116,chr22,42214837,T,C,NM_005650_3:TCF20:missense:157/1960(Thr->Ala),TCF20,1960


In [34]:
de_novo_coding_vars_expanded[["CHROM", "POS", "REF", "ALT"]].drop_duplicates()

,CHROM,POS,REF,ALT
0,chr1,61352564,C,T
1,chr2,1917307,A,G
2,chr2,156329364,A,G
4,chr4,105234731,G,T
5,chr9,37033986,C,G
...,...,...,...,...
112,chr18,55228885,G,A
113,chr18,55228994,G,A
114,chr21,38445461,G,C
115,chr22,42213189,C,T
